# 99 — Day 1 Spike

Pick the base VLM for Day 2 by smoke-testing 3 Qwen3-VL Instruct variants (2B / 4B / 8B) and benchmarking them against GPT-4o-mini on the locked 100-row test set.

## What this notebook decides

- Which Qwen3-VL variant loads cleanly on a Colab T4 in 4-bit.
- Whether any Qwen variant is competitive with GPT-4o-mini on parse rate, brand / category / condition / color accuracy, and price MAPE.
- The model ID to lock for the Day 2 QLoRA fine-tune.

## Out of scope (everything below is Day 2+)

- QLoRA fine-tuning.
- DINOv2 visible-flaw head.
- Quantile price head with pinball loss.
- Sell-likelihood head.
- Frontend, FastAPI backend, SQLite logging.

This notebook is **zero-shot only**. No model weights are trained.

## Inputs / outputs

- Inputs: `data/vinted_clothing_v1_5_full.parquet`, `data/kleinanzeigen_clothing_v1.parquet`
- Outputs: prediction JSONs in `results/spike/`, locked test set in `data/splits/spike_test.parquet`
- Decision gate: see brief §2.4. Verdict goes in cell 14.

## Idempotency

Every cell is safe to re-run. Predictions are cached to disk; the OpenAI key is only prompted on first run.


In [ ]:
# Pinned versions for Colab T4 stability. Re-running is a no-op once installed.
%pip install --quiet \
    "transformers>=4.51" \
    "accelerate>=1.0" \
    "bitsandbytes>=0.44" \
    "openai>=1.55" \
    "pandas>=2.2" \
    "pyarrow>=16.0" \
    "pillow>=10.3" \
    "scikit-learn>=1.5" \
    "tabulate>=0.9"


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/mchlkan/Advanced_ML.git"
REPO_DIR = Path("/content/Advanced_ML")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    # Best-effort pull so the spike runs against latest data_prep.py
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"cwd        : {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")


In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"Device        : {p.name}")
    print(f"VRAM total    : {p.total_memory / 1e9:.1f} GB")
    print(f"CUDA          : {torch.version.cuda}")


In [ ]:
import json
import shutil
from pathlib import Path
import pandas as pd

VINTED_FN = "vinted_clothing_v1_5_full.parquet"
KA_FN     = "kleinanzeigen_clothing_v1.parquet"
DATA_DIR    = Path("data")
SPLITS_DIR  = Path("data/splits")
RESULTS_DIR = Path("results/spike")
DATA_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Bring data in from Google Drive on first run if not already local.
if not (DATA_DIR / VINTED_FN).exists() or not (DATA_DIR / KA_FN).exists():
    print("Data files not found locally — mounting Google Drive...")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA = Path("/content/drive/MyDrive/Resell_Copilot_data")
    for fn in (VINTED_FN, KA_FN):
        dst = DATA_DIR / fn
        if dst.exists():
            continue
        src = DRIVE_DATA / fn
        if not src.exists():
            raise FileNotFoundError(
                f"{fn} not found in {DRIVE_DATA}. "
                f"Place both parquets there or copy them to {DATA_DIR}/ manually."
            )
        shutil.copy(src, dst)
        print(f"  copied {fn} from Drive")

import data_prep

vt = data_prep.apply_filters(data_prep.load_vinted(str(DATA_DIR / VINTED_FN)), "vinted")
ka = data_prep.apply_filters(data_prep.load_kleinanzeigen(str(DATA_DIR / KA_FN)), "kleinanzeigen")
vt = data_prep.filter_vinted_de_only(vt)
combined = data_prep.build_combined(vt, ka)
sampled = data_prep.stratified_sample(
    combined, n_per_platform=300,
    strata_cols=["category_name", "condition"], seed=SEED,
)
print(f"Vinted (de, filtered): {len(vt):,}")
print(f"KA      (filtered)   : {len(ka):,}")
print(f"Combined sampled     : {len(sampled):,}  {sampled['platform'].value_counts().to_dict()}")

ids_path     = SPLITS_DIR / "spike_test_ids.json"
parquet_path = SPLITS_DIR / "spike_test.parquet"

if parquet_path.exists() and ids_path.exists():
    test_df = pd.read_parquet(parquet_path)
    print(f"Reused locked test set: {parquet_path} ({len(test_df)} rows)")
else:
    train_df, test_df = data_prep.train_test_split_by_id(
        sampled, test_size=100,
        strata_cols=["category_name", "condition", "platform"], seed=SEED,
    )
    test_df.to_parquet(parquet_path, index=False)
    with open(ids_path, "w") as f:
        json.dump([int(x) for x in test_df["id"].tolist()], f)
    print(f"Wrote test set to {parquet_path} and {ids_path}")

print(f"Test by platform: {test_df['platform'].value_counts().to_dict()}")


In [ ]:
import re
import json
from typing import Optional

SCHEMA_FIELDS = ["brand", "category", "condition", "color", "size",
                 "title", "description", "price_eur"]


def parse_model_output(raw: str) -> Optional[dict]:
    """Parse a model's text output into a dict over SCHEMA_FIELDS.

    Tolerant: strips ```json / ``` fences and surrounding prose, finds the
    first matching {...} block, retries with trailing-comma cleanup if the
    JSON is slightly malformed. Missing fields become None. Returns None if
    no JSON object is parseable.
    """
    if not raw:
        return None
    text = raw.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```\s*$", "", text)
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    end = -1
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                end = i
                break
    if end == -1:
        return None
    snippet = text[start:end + 1]
    try:
        obj = json.loads(snippet)
    except json.JSONDecodeError:
        cleaned = re.sub(r",\s*([}\]])", r"\1", snippet)
        try:
            obj = json.loads(cleaned)
        except json.JSONDecodeError:
            return None
    if not isinstance(obj, dict):
        return None
    return {k: obj.get(k) for k in SCHEMA_FIELDS}


# self-tests
for sample in [
    '```json\n{"brand": "Nike", "category": "tshirts"}\n```',
    'Hier das JSON: {"brand": "Zara", "price_eur": 20.0} — fertig.',
    '{"brand": null, "category": "jackets",}',
    'no json here at all',
]:
    print(parse_model_output(sample))


In [ ]:
VINTED_CATEGORIES = ["jackets", "jeans", "tshirts", "sneakers"]
KA_CATEGORIES     = ["Damenbekleidung", "Herrenbekleidung", "Damenschuhe", "Herrenschuhe"]
CONDITION_VALUES  = ["neu mit etikett", "neu", "sehr gut", "gut"]


def get_prompt(platform: str) -> str:
    """Return the German VLM prompt template for a given platform."""
    if platform == "vinted":
        platform_name = "Vinted"
        cats = VINTED_CATEGORIES
    elif platform == "kleinanzeigen":
        platform_name = "Kleinanzeigen"
        cats = KA_CATEGORIES
    else:
        raise ValueError(f"Unknown platform: {platform}")
    cats_str = json.dumps(cats, ensure_ascii=False)
    cond_str = json.dumps(CONDITION_VALUES, ensure_ascii=False)
    return (
        f"Du bist ein Experte für {platform_name}-Inserate. "
        f"Analysiere das Foto dieses Kleidungsstücks und gib AUSSCHLIESSLICH ein "
        f"einzelnes JSON-Objekt zurück (keine Erklärungen, kein Markdown, kein Code-Fence).\n\n"
        f"Format:\n"
        f"{{\n"
        f'  "brand": <Markenname als String, oder null>,\n'
        f'  "category": <eine von {cats_str}>,\n'
        f'  "condition": <eine von {cond_str}>,\n'
        f'  "color": <Farbe auf Deutsch>,\n'
        f'  "size": <Größe als String, oder null>,\n'
        f'  "title": <Listing-Titel auf Deutsch>,\n'
        f'  "description": <Beschreibung auf Deutsch, 2-3 Sätze>,\n'
        f'  "price_eur": <Verkaufspreis in EUR als Zahl>\n'
        f"}}\n\n"
        f"Antworte nur mit dem JSON-Objekt."
    )


print("--- VINTED PROMPT ---")
print(get_prompt("vinted"))
print()
print("--- KLEINANZEIGEN PROMPT ---")
print(get_prompt("kleinanzeigen"))


In [ ]:
import gc
import io
import time
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

_LOADED = {"name": None, "processor": None, "model": None}


def _free_loaded():
    if _LOADED["model"] is not None:
        del _LOADED["model"]
        del _LOADED["processor"]
    _LOADED.update(name=None, processor=None, model=None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def load_qwen(model_id: str):
    """Load (or reuse) a Qwen3-VL Instruct model in 4-bit. Frees prior model."""
    if _LOADED["name"] == model_id:
        return _LOADED["processor"], _LOADED["model"]
    _free_loaded()
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        quantization_config=bnb_cfg,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    _LOADED.update(name=model_id, processor=processor, model=model)
    return processor, model


def pil_from_row(row) -> Image.Image:
    """Decode the HuggingFace image dict from a row into a PIL RGB image."""
    img = row["image"]
    if isinstance(img, dict) and img.get("bytes"):
        return Image.open(io.BytesIO(img["bytes"])).convert("RGB")
    if isinstance(img, (bytes, bytearray)):
        return Image.open(io.BytesIO(img)).convert("RGB")
    raise ValueError("Row has no decodable image bytes.")


def run_qwen_inference(processor, model, image: Image.Image, platform: str,
                        max_new_tokens: int = 384):
    """Run a single (image, platform) inference. Returns (raw_text, latency_seconds)."""
    prompt = get_prompt(platform)
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    latency = time.time() - t0
    new_tokens = out_ids[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return text.strip(), latency


In [ ]:
QWEN_MODELS = [
    "Qwen/Qwen3-VL-2B-Instruct",
    "Qwen/Qwen3-VL-4B-Instruct",
    "Qwen/Qwen3-VL-8B-Instruct",
]

LOAD_RESULTS = {}  # model_id -> "ok" | failure reason

smoke_rows = test_df.sample(5, random_state=SEED).reset_index(drop=True)

for mid in QWEN_MODELS:
    print(f"\n{'='*70}\n  Smoke-testing {mid}\n{'='*70}")
    try:
        processor, model = load_qwen(mid)
    except Exception as e:
        msg = f"load_failed: {type(e).__name__}: {e}"
        print(f"  {msg}")
        LOAD_RESULTS[mid] = msg
        _free_loaded()
        continue
    try:
        for i, row in smoke_rows.iterrows():
            img = pil_from_row(row)
            raw, lat = run_qwen_inference(processor, model, img, row["platform"])
            preview = raw[:180].replace("\n", " ")
            print(f"  [{i+1}/{len(smoke_rows)}] platform={row['platform']:<14} {lat:.1f}s")
            print(f"    {preview}{'...' if len(raw) > 180 else ''}")
        LOAD_RESULTS[mid] = "ok"
    except Exception as e:
        msg = f"inference_failed: {type(e).__name__}: {e}"
        print(f"  {msg}")
        LOAD_RESULTS[mid] = msg

_free_loaded()
print("\nSmoke-test summary:")
for k, v in LOAD_RESULTS.items():
    print(f"  {k}: {v}")


In [ ]:
def model_slug(model_id: str) -> str:
    return model_id.split("/")[-1].lower()


PERF = {}  # slug -> {"latencies": [...], "peak_vram_gb": float}


def run_full_qwen_eval(model_id: str, df: pd.DataFrame):
    slug = model_slug(model_id)
    out_path = RESULTS_DIR / f"{slug}_predictions.json"
    if out_path.exists():
        records = json.loads(out_path.read_text())
        PERF[slug] = {
            "latencies": [r["latency_s"] for r in records if "latency_s" in r],
            "peak_vram_gb": next((r.get("_peak_vram_gb_at_run") for r in records
                                  if "_peak_vram_gb_at_run" in r), None),
        }
        print(f"  cached: {out_path}  ({len(records)} rows)")
        return out_path
    if LOAD_RESULTS.get(model_id) != "ok":
        print(f"  skipped ({LOAD_RESULTS.get(model_id, 'not smoke-tested')})")
        return None

    processor, model = load_qwen(model_id)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    records = []
    for i, row in df.iterrows():
        try:
            img = pil_from_row(row)
            raw, lat = run_qwen_inference(processor, model, img, row["platform"])
            parsed = parse_model_output(raw)
        except Exception as e:
            raw, lat, parsed = f"ERROR: {type(e).__name__}: {e}", 0.0, None
        records.append({
            "id": int(row["id"]),
            "platform": row["platform"],
            "raw": raw,
            "parsed": parsed,
            "ground_truth": {
                "brand":       row.get("brand"),
                "category":    row.get("category_name"),
                "condition":   row.get("condition"),
                "color":       row.get("color"),
                "size":        row.get("size"),
                "title":       row.get("title"),
                "description": row.get("description"),
                "price_eur":   float(row["price"]) if pd.notna(row.get("price")) else None,
            },
            "latency_s": lat,
        })
        if (i + 1) % 20 == 0:
            print(f"    {i+1}/{len(df)}")

    peak_vram_gb = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
    if records:
        records[0]["_peak_vram_gb_at_run"] = peak_vram_gb
    out_path.write_text(json.dumps(records, indent=2, default=str, ensure_ascii=False))
    PERF[slug] = {"latencies": [r["latency_s"] for r in records], "peak_vram_gb": peak_vram_gb}
    print(f"  wrote {out_path}  (peak VRAM {peak_vram_gb:.2f} GB)")
    return out_path


for mid in QWEN_MODELS:
    print(f"\nFull eval: {mid}")
    run_full_qwen_eval(mid, test_df)

_free_loaded()


In [ ]:
import base64
import getpass
from openai import OpenAI

GPT_MODEL    = "gpt-4o-mini"
GPT_HARD_CAP = 100   # max API calls; safety net
GPT_OUT      = RESULTS_DIR / "gpt4omini_predictions.json"

if GPT_OUT.exists():
    print(f"cached: {GPT_OUT}")
    cached = json.loads(GPT_OUT.read_text())
    PERF["gpt-4o-mini"] = {"latencies": [r["latency_s"] for r in cached], "peak_vram_gb": 0.0}
else:
    api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OPENAI_API_KEY: ")
    client = OpenAI(api_key=api_key)

    def img_to_data_url(img):
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

    records = []
    n_calls = 0
    for i, row in test_df.iterrows():
        if n_calls >= GPT_HARD_CAP:
            print(f"  hit hard cap of {GPT_HARD_CAP}, stopping")
            break
        try:
            img = pil_from_row(row)
            t0 = time.time()
            resp = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "image_url",
                         "image_url": {"url": img_to_data_url(img)}},
                        {"type": "text", "text": get_prompt(row["platform"])},
                    ],
                }],
                max_tokens=400,
                temperature=0.0,
            )
            lat = time.time() - t0
            raw = resp.choices[0].message.content
            n_calls += 1
            parsed = parse_model_output(raw)
        except Exception as e:
            raw, lat, parsed = f"ERROR: {type(e).__name__}: {e}", 0.0, None

        records.append({
            "id": int(row["id"]),
            "platform": row["platform"],
            "raw": raw,
            "parsed": parsed,
            "ground_truth": {
                "brand":       row.get("brand"),
                "category":    row.get("category_name"),
                "condition":   row.get("condition"),
                "color":       row.get("color"),
                "size":        row.get("size"),
                "title":       row.get("title"),
                "description": row.get("description"),
                "price_eur":   float(row["price"]) if pd.notna(row.get("price")) else None,
            },
            "latency_s": lat,
        })
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(test_df)}  (api_calls={n_calls})")

    GPT_OUT.write_text(json.dumps(records, indent=2, default=str, ensure_ascii=False))
    PERF["gpt-4o-mini"] = {"latencies": [r["latency_s"] for r in records], "peak_vram_gb": 0.0}
    print(f"  wrote {GPT_OUT}  ({n_calls} API calls)")


In [ ]:
def _norm(v):
    return None if v is None else str(v).lower().strip()


def compute_metrics(records: list) -> dict:
    n = len(records)
    if n == 0:
        return {"n": 0}

    parsed_ok = sum(1 for r in records if r["parsed"] is not None)
    correct = {"brand": 0, "category": 0, "condition": 0, "color": 0}
    valid   = {"brand": 0, "category": 0, "condition": 0, "color": 0}
    abs_pct_errs = []

    for r in records:
        p, gt = r["parsed"], r["ground_truth"]
        if p is None:
            continue
        for f in ("brand", "category", "condition", "color"):
            true = gt.get(f)
            if true is None:
                continue
            valid[f] += 1
            if _norm(p.get(f)) == _norm(true):
                correct[f] += 1
        try:
            pred_price = float(p.get("price_eur"))
            true_price = float(gt.get("price_eur"))
            if true_price > 0:
                abs_pct_errs.append(abs(pred_price - true_price) / true_price)
        except (TypeError, ValueError):
            pass

    return {
        "n":            n,
        "parse_rate":   parsed_ok / n,
        "brand_acc":    (correct["brand"]     / valid["brand"])     if valid["brand"]     else None,
        "category_acc": (correct["category"]  / valid["category"])  if valid["category"]  else None,
        "condition_acc":(correct["condition"] / valid["condition"]) if valid["condition"] else None,
        "color_acc":    (correct["color"]     / valid["color"])     if valid["color"]     else None,
        "price_mape":   float(np.mean(abs_pct_errs)) if abs_pct_errs else None,
        "price_n":      len(abs_pct_errs),
    }


PRED_FILES = []
for mid in QWEN_MODELS:
    p = RESULTS_DIR / f"{model_slug(mid)}_predictions.json"
    if p.exists():
        PRED_FILES.append((model_slug(mid), p))
if (RESULTS_DIR / "gpt4omini_predictions.json").exists():
    PRED_FILES.append(("gpt-4o-mini", RESULTS_DIR / "gpt4omini_predictions.json"))

rows = []
for name, path in PRED_FILES:
    rows.append({"model": name, **compute_metrics(json.loads(path.read_text()))})

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_markdown(index=False, floatfmt=".3f"))


In [ ]:
perf_rows = []
for slug, perf in PERF.items():
    lats = perf.get("latencies", [])
    perf_rows.append({
        "model":            slug,
        "n_calls":          len(lats),
        "median_latency_s": float(np.median(lats))         if lats else None,
        "p95_latency_s":    float(np.percentile(lats, 95)) if lats else None,
        "peak_vram_gb":     perf.get("peak_vram_gb"),
    })
perf_df = pd.DataFrame(perf_rows)
print(perf_df.to_markdown(index=False, floatfmt=".2f"))


## Verdict — fill in by hand after running

- **Locked model for Day 2:** _<model id>_
- **Why:** _<brief reason — best parse rate / best price MAPE / load-able on T4 / fastest>_
- **GPT-4o-mini comparison:** _<we win / tie / lose, on which fields>_
- **Notable failure modes seen in raw outputs:** _<e.g. prose around JSON, wrong language, hallucinated brands>_
- **Decision per brief §2.4:** _<commit / diagnose / reframe>_

If no Qwen variant beats GPT-4o-mini on at least one of {brand, category, price MAPE}, **diagnose before scaling up** — likely a prompt or hyperparameter issue, not a base-model issue.
